In [ ]:
import os
import pandas as pd

In [ ]:
# ========== 需要提取的情景与时间 ==========
ssps = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]
times = [2020, 2040, 2060, 2080, 2100]  # 2020 全 0；其余提取

In [ ]:
# ========== 路径配置（按你给的写法） ==========
resolution = "10km"  # TODO: '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km'
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)

# ✅ 输出目录：base_path/outputs/path_name
out_dir = os.path.join(base_path, "outputs", path_name)
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

PREDICTION_DIR = Path(base_path) / "code" / "5_gwr_model_prediction"
POINTS_DIR = Path(data_base_path) / "Extracted_HAVE_future" / f"Points_China_all_{resolution}"
PROVINCE_SHP = Path(base_path) / "data" / "Administrative_divisions_of_china" / "no_TW_AM_HK" / "china_pro2_no_TW_AM_HK.shp"

PROVINCE_NAME_MAP = {
    510000: "Sichuan",
    460000: "Hainan",
    410000: "Henan",
    310000: "Shanghai",
    120000: "Tianjin",
    110000: "Beijing",
    540000: "Tibet",
    630000: "Qinghai",
    640000: "Ningxia",
    350000: "Fujian",
    620000: "Gansu",
    530000: "Yunnan",
    320000: "Jiangsu",
    330000: "Zhejiang",
    230000: "Heilongjiang",
    220000: "Jilin",
    150000: "Inner Mongolia",
    610000: "Shaanxi",
    340000: "Anhui",
    500000: "Chongqing",
    650000: "Xinjiang",
    130000: "Hebei",
    420000: "Hubei",
    440000: "Guangdong",
    210000: "Liaoning",
    370000: "Shandong",
    520000: "Guizhou",
    360000: "Jiangxi",
    430000: "Hunan",
    450000: "Guangxi",
    140000: "Shanxi",
}


def prediction_pkl_path(ssp: str, ssp_time: str) -> Path:
    return PREDICTION_DIR / f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl"


def point_csv_path(ssp: str) -> Path:
    return POINTS_DIR / f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv"


def pick_lon_lat_cols(df: pd.DataFrame):
    lon_candidates = ["Longitude", "longitude", "LON", "lon", "X", "x"]
    lat_candidates = ["Latitude", "latitude", "LAT", "lat", "Y", "y"]
    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    if lon_col is None or lat_col is None:
        raise KeyError(f"无法在 df 中找到经纬度列。可选列名示例：{lon_candidates} / {lat_candidates}")
    return lon_col, lat_col


def load_points_lonlat(ssp: str):
    csv_path = point_csv_path(ssp)
    if not csv_path.exists():
        raise FileNotFoundError(f"找不到点文件：{csv_path}")

    df = pd.read_csv(csv_path)
    lon_col, lat_col = pick_lon_lat_cols(df)
    out = df[[lon_col, lat_col]].copy()
    out.rename(columns={lon_col: "Longitude", lat_col: "Latitude"}, inplace=True)
    return out, csv_path


def load_prediction_scores(ssp: str, ssp_time: str):
    pkl_path = prediction_pkl_path(ssp, ssp_time)
    if not pkl_path.exists():
        raise FileNotFoundError(f"找不到预测结果 pkl：{pkl_path}")

    with pkl_path.open("rb") as f:
        saved = pickle.load(f)

    if "y_pred_gwr" not in saved:
        raise KeyError(f"{pkl_path} 中缺少 y_pred_gwr")

    scores = np.asarray(saved["y_pred_gwr"], dtype=float).reshape(-1)
    return scores, pkl_path


def add_xy_key(df: pd.DataFrame, nd: int = 6) -> pd.DataFrame:
    out = df.copy()
    out["lon_r"] = pd.to_numeric(out["Longitude"], errors="coerce").round(nd)
    out["lat_r"] = pd.to_numeric(out["Latitude"], errors="coerce").round(nd)
    out = out.dropna(subset=["lon_r", "lat_r"])
    return out


def build_province_raw_change_table(
    current_points: pd.DataFrame,
    current_scores: np.ndarray,
    hist_points: pd.DataFrame,
    hist_scores: np.ndarray,
) -> tuple[pd.DataFrame, dict]:
    current_scores = np.asarray(current_scores, dtype=float).reshape(-1)
    hist_scores = np.asarray(hist_scores, dtype=float).reshape(-1)

    if len(current_points) != len(current_scores):
        raise ValueError(
            "当前点 CSV 行数与 y_pred_gwr 长度不一致：\n"
            f"len(current_points)={len(current_points)}, len(current_scores)={len(current_scores)}"
        )
    if len(hist_points) != len(hist_scores):
        raise ValueError(
            "2020(hist) 点 CSV 行数与 y_pred_gwr 长度不一致：\n"
            f"len(hist_points)={len(hist_points)}, len(hist_scores)={len(hist_scores)}"
        )

    df_cur = current_points.copy()
    df_cur["GWR_Score_Current"] = current_scores

    df_2020 = hist_points.copy()
    df_2020["GWR_Score_2020"] = hist_scores

    df_cur["GWR_Score_Current"] = pd.to_numeric(df_cur["GWR_Score_Current"], errors="coerce")
    df_2020["GWR_Score_2020"] = pd.to_numeric(df_2020["GWR_Score_2020"], errors="coerce")

    df_cur_k = add_xy_key(df_cur, nd=6)
    df_2020_k = add_xy_key(df_2020, nd=6)

    cur_pts = df_cur_k.groupby(["lon_r", "lat_r"], as_index=False).agg(
        GWR_Score_Current=("GWR_Score_Current", "mean")
    )
    hist_pts = df_2020_k.groupby(["lon_r", "lat_r"], as_index=False).agg(
        GWR_Score_2020=("GWR_Score_2020", "mean")
    )

    aligned = cur_pts.merge(hist_pts, on=["lon_r", "lat_r"], how="inner")
    if len(aligned) == 0:
        raise ValueError("对齐后为 0 个点：请检查两套点的坐标是否一致。")

    prov_raw = gpd.read_file(PROVINCE_SHP)
    adcode_col = next((c for c in prov_raw.columns if str(c).strip().lower() == "adcode99"), None)
    if adcode_col is None:
        raise KeyError(f"shp 中找不到 ADCODE99 列。当前列名：{list(prov_raw.columns)}")

    prov = prov_raw[[adcode_col, "geometry"]].copy().rename(columns={adcode_col: "ADCODE99"})
    if prov.crs is None:
        prov = prov.set_crs("EPSG:4326")

    g_aligned = gpd.GeoDataFrame(
        aligned.copy(),
        geometry=gpd.points_from_xy(aligned["lon_r"], aligned["lat_r"]),
        crs="EPSG:4326",
    ).to_crs(prov.crs)

    joined = gpd.sjoin(g_aligned, prov, how="left", predicate="within")
    joined = joined.dropna(subset=["ADCODE99"]).copy()
    joined["ADCODE99"] = pd.to_numeric(joined["ADCODE99"], errors="coerce").astype("Int64")
    joined["NAME_EN_JX"] = joined["ADCODE99"].map(PROVINCE_NAME_MAP)
    joined = joined.dropna(subset=["NAME_EN_JX"]).copy()

    if joined.empty:
        raise ValueError("成功对齐了点，但没有点成功匹配到省级行政区。")

    stats = (
        joined.groupby(["ADCODE99", "NAME_EN_JX"], dropna=False)
        .agg(
            n_points=("GWR_Score_Current", "size"),
            mean_score_2020=("GWR_Score_2020", "mean"),
            mean_score_current=("GWR_Score_Current", "mean"),
            median_score_2020=("GWR_Score_2020", "median"),
            median_score_current=("GWR_Score_Current", "median"),
        )
        .reset_index()
    )

    stats["delta_mean"] = stats["mean_score_current"] - stats["mean_score_2020"]
    stats["delta_median"] = stats["median_score_current"] - stats["median_score_2020"]
    stats["change_rate"] = np.where(
        np.abs(stats["mean_score_2020"]) > 1e-12,
        stats["delta_mean"] / np.abs(stats["mean_score_2020"]),
        np.nan,
    )
    stats["change_rate_pct"] = stats["change_rate"] * 100.0

    china_mean_2020 = float(joined["GWR_Score_2020"].mean())
    china_mean_cur = float(joined["GWR_Score_Current"].mean())
    china_delta = china_mean_cur - china_mean_2020
    china_base = abs(china_mean_2020)
    china_change_rate = (china_delta / china_base) if china_base > 1e-12 else np.nan

    china = pd.DataFrame([{
        "ADCODE99": 100000,
        "NAME_EN_JX": "China",
        "n_points": int(len(joined)),
        "mean_score_2020": china_mean_2020,
        "mean_score_current": china_mean_cur,
        "delta_mean": float(china_delta),
        "change_rate": float(china_change_rate) if not pd.isna(china_change_rate) else np.nan,
        "change_rate_pct": float(china_change_rate * 100.0) if not pd.isna(china_change_rate) else np.nan,
        "median_score_2020": float(joined["GWR_Score_2020"].median()),
        "median_score_current": float(joined["GWR_Score_Current"].median()),
        "delta_median": float(joined["GWR_Score_Current"].median() - joined["GWR_Score_2020"].median()),
    }])

    out = pd.concat([china, stats], ignore_index=True)
    out = out.sort_values(by=["change_rate"], ascending=False, na_position="last").reset_index(drop=True)
    out = out[[
        "ADCODE99", "NAME_EN_JX",
        "n_points",
        "mean_score_2020", "mean_score_current", "delta_mean",
        "change_rate", "change_rate_pct",
        "median_score_2020", "median_score_current", "delta_median",
    ]]

    summary = {
        "aligned_current_points": int(len(cur_pts)),
        "aligned_hist_points": int(len(hist_pts)),
        "matched_points": int(len(aligned)),
        "province_matched_points": int(len(joined)),
    }
    return out, summary


hist_points, hist_csv_path = load_points_lonlat("hist")
hist_scores, hist_pkl_path = load_prediction_scores("hist", "2020")

print(f"2020(hist) 点文件: {hist_csv_path}")
print(f"2020(hist) 预测 pkl: {hist_pkl_path}")
print(f"2020(hist) 点数 / 预测长度: {len(hist_points)} / {len(hist_scores)}")

scenario_points = {}
for ssp in ssps:
    points_df, csv_path = load_points_lonlat(ssp)
    scenario_points[ssp] = points_df
    print(f"{ssp} 点文件: {csv_path} | 点数: {len(points_df)}")

province_change_tables = {}
result_numeric = pd.DataFrame({"Time": times})
for ssp in ssps:
    result_numeric[ssp] = 0.0

for t in [2040, 2060, 2080, 2100]:
    ssp_time = str(t)
    for ssp in ssps:
        current_scores, current_pkl_path = load_prediction_scores(ssp, ssp_time)
        province_change_df, summary = build_province_raw_change_table(
            current_points=scenario_points[ssp],
            current_scores=current_scores,
            hist_points=hist_points,
            hist_scores=hist_scores,
        )

        province_change_tables[(ssp, ssp_time)] = province_change_df

        scenario_out_dir = Path(out_dir) / ssp / ssp_time
        scenario_out_dir.mkdir(parents=True, exist_ok=True)
        csv_name = f"province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_{ssp}_{ssp_time}.csv"
        csv_path = scenario_out_dir / csv_name
        province_change_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        china_row = province_change_df.loc[province_change_df["NAME_EN_JX"].astype(str).str.strip().eq("China")]
        if china_row.empty:
            raise ValueError(f"{ssp}-{ssp_time} 的省级变化表中没有 China 行。")

        china_change_pct = float(china_row.iloc[0]["change_rate_pct"])
        result_numeric.loc[result_numeric["Time"] == t, ssp] = china_change_pct

        print(
            f"{ssp}-{ssp_time}: China change_rate_pct = {china_change_pct:.2f}% | "
            f"matched_points = {summary['matched_points']} | province_points = {summary['province_matched_points']} | "
            f"csv = {csv_path}"
        )

result = result_numeric.copy()
for ssp in ssps:
    result[ssp] = result[ssp].map(lambda v: f"{float(v):.2f}%")

out_csv = os.path.join(out_dir, "risk_pre_ssp_ssp_time.csv")
out_csv_numeric = os.path.join(out_dir, "risk_pre_ssp_ssp_time_numeric.csv")

result.to_csv(out_csv, index=False, sep="\t", encoding="utf-8-sig")
result_numeric.to_csv(out_csv_numeric, index=False, encoding="utf-8-sig")

print(f"已保存字符串版结果：{out_csv}")
print(f"已保存数值版结果：{out_csv_numeric}")
print(result)
display(result_numeric)


绘图

In [ ]:
# ========== Heatmap + 边际均值条形（原始 GWR 风险变化率；TNR=9；无colorbar；边框灰；所有文字黑；所有刻度线灰；右侧y轴无刻度线）==========
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker  # ✅ 新增：用于彻底移除刻度

ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]

# --- 把 "10.45%" -> 10.45（数值，单位：%）---
plot_df = result.copy()
for c in ssp_cols:
    plot_df[c] = plot_df[c].astype(str).str.replace("%", "", regex=False).astype(float)

times = plot_df["Time"].astype(int).tolist()
Z = plot_df[ssp_cols].to_numpy().T  # (SSP, Time)
ssp_labels = [c.upper() for c in ssp_cols]

mean_by_time = plot_df[ssp_cols].mean(axis=1).to_numpy()  # (Time,)
mean_by_ssp  = plot_df[ssp_cols].mean(axis=0).to_numpy()  # (SSP,)

# --- 统一颜色规范 ---
SPINE_GRAY = "#7A7A7A"   # 边框
TICK_GRAY  = "#7A7A7A"   # 刻度线
TEXT_BLACK = "black"     # 所有文字黑色

# --- 风格：Times New Roman + 字体 9（文字默认黑色） ---
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "text.color": TEXT_BLACK,
    "axes.labelcolor": TEXT_BLACK,
    "xtick.color": TEXT_BLACK,   # 刻度文字黑
    "ytick.color": TEXT_BLACK,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# --- 颜色：值域固定 0~37% ---
cmap = LinearSegmentedColormap.from_list("peach_to_blue", ["#FFF4EE", "#FB7D5D"], N=256)
vmin, vmax = 0.0, 37.0
norm = Normalize(vmin=vmin, vmax=vmax)

# 边际柱颜色：按均值映射到同一 cmap/norm
top_colors   = cmap(norm(mean_by_time))
right_colors = cmap(norm(mean_by_ssp))

# --- 版面：上方均值柱 + 主热力图 + 右侧均值条 ---
fig = plt.figure(figsize=(3.35, 2.45), dpi=300)
gs = GridSpec(
    2, 2, figure=fig,
    width_ratios=[5.0, 1.4],
    height_ratios=[1.2, 5.0],
    wspace=0.15, hspace=0.12
)

ax_top   = fig.add_subplot(gs[0, 0])
ax_main  = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1])   # ✅ 不再 sharey，避免影响主图
ax_blank = fig.add_subplot(gs[0, 1]); ax_blank.axis("off")

# --- 主热力图 ---
ax_main.imshow(
    Z, cmap=cmap, norm=norm,
    aspect="auto", origin="upper", interpolation="nearest"
)

xpos = np.arange(len(times))
ypos = np.arange(len(ssp_labels))

ax_main.set_xticks(xpos)
ax_main.set_xticklabels([str(t) for t in times], color=TEXT_BLACK)
ax_main.set_yticks(ypos)
ax_main.set_yticklabels(ssp_labels, color=TEXT_BLACK)  # ✅ SSP1-5 保证显示

ax_main.set_xlabel("Year", color=TEXT_BLACK)
ax_main.set_ylabel("Scenario", color=TEXT_BLACK)

# 细网格分隔（表格感）
ax_main.set_xticks(np.arange(-.5, len(times), 1), minor=True)
ax_main.set_yticks(np.arange(-.5, len(ssp_labels), 1), minor=True)
ax_main.grid(which="minor", linewidth=0.6, alpha=0.35)
ax_main.tick_params(which="minor", bottom=False, left=False)

# 单元格标注：全部黑色
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        val = float(Z[i, j])
        ax_main.text(j, i, f"{val:.2f}%", ha="center", va="center", color=TEXT_BLACK, fontsize=8)

# 主图边框：四边灰色
for side in ["top", "right", "bottom", "left"]:
    ax_main.spines[side].set_visible(True)
    ax_main.spines[side].set_linewidth(0.8)
    ax_main.spines[side].set_color(SPINE_GRAY)

# --- 上方：各年份均值（颜色映射；y轴刻度=0/10/20；y=0 横轴线；x轴无刻度线） ---
ax_top.bar(xpos, mean_by_time, width=0.7, edgecolor="none", color=top_colors)
ax_top.set_xlim(-0.5, len(times) - 0.5)
ax_top.set_ylabel("Mean (%)", color=TEXT_BLACK)

ax_top.set_xticks(xpos)
ax_top.tick_params(axis="x", labelbottom=False, length=0, color=TICK_GRAY, labelcolor=TEXT_BLACK)

ax_top.set_yticks([0, 15, 30])
ax_top.set_ylim(0, 30)

ax_top.spines["bottom"].set_visible(True)
ax_top.spines["bottom"].set_linewidth(0.8)
ax_top.spines["bottom"].set_position(("data", 0))
ax_top.spines["bottom"].set_color(SPINE_GRAY)

ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)
ax_top.spines["left"].set_visible(True)
ax_top.spines["left"].set_linewidth(0.8)
ax_top.spines["left"].set_color(SPINE_GRAY)

ax_top.grid(True, axis="y", linewidth=0.5, alpha=0.25)

# --- 右侧：各SSP跨年份均值（颜色映射；x轴刻度=0/10/20；彻底移除y轴刻度线/标签） ---
ax_right.barh(ypos, mean_by_ssp, height=0.7, edgecolor="none", color=right_colors)
ax_right.set_xlabel("Mean (%)", color=TEXT_BLACK)

# ✅ 对齐主图y轴范围（含反向），保证条形与热力图行严格对齐
ax_right.set_ylim(ax_main.get_ylim())

# ✅ 彻底移除右侧小图 y 轴刻度线与标签
ax_right.set_yticks([])
ax_right.tick_params(axis="y", which="both", left=False, right=False, labelleft=False, labelright=False, length=0)

ax_right.set_xticks([0, 15, 30])
ax_right.set_xlim(0, 30)
ax_right.tick_params(axis="x", which="both", color=TICK_GRAY, labelcolor=TEXT_BLACK)

# 右侧小图边框（left/bottom灰色；top/right不显示）
ax_right.spines["top"].set_visible(False)
ax_right.spines["right"].set_visible(False)
ax_right.spines["left"].set_visible(True)
ax_right.spines["bottom"].set_visible(True)
ax_right.spines["left"].set_linewidth(0.8)
ax_right.spines["bottom"].set_linewidth(0.8)
ax_right.spines["left"].set_color(SPINE_GRAY)
ax_right.spines["bottom"].set_color(SPINE_GRAY)

ax_right.grid(True, axis="x", linewidth=0.5, alpha=0.25)

# --- 所有刻度线灰色；文字黑色（主图/上图/右图一致） ---
for ax in [ax_top, ax_main, ax_right]:
    ax.tick_params(axis="both", which="major", color=TICK_GRAY, labelcolor=TEXT_BLACK)
    ax.tick_params(axis="both", which="minor", color=TICK_GRAY, labelcolor=TEXT_BLACK)


# --- 只保存为 SVG（不保存 PNG/PDF）---
fig_path_svg = os.path.join(out_dir, "risk_pre_ssp_ssp_time_b.svg")
plt.savefig(fig_path_svg, format="svg", bbox_inches="tight")
plt.show()
plt.close(fig)

print("Figure saved:")
print(fig_path_svg)



绘制趋势图

In [ ]:
# ========== 绘图：Nature 风格（简洁、出版级）+ 原始 GWR 风险变化率不确定性误差带（随时间累积）==========
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy.interpolate import UnivariateSpline
from matplotlib.ticker import MultipleLocator
from matplotlib import font_manager

# 如果你这里是从文件读回来的，就取消注释：
# result = pd.read_csv(out_csv, sep="\t")

# --- 把 "10.45%" -> 10.45（数值，单位：%）---
plot_df = result.copy()
ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]
for c in ssp_cols:
    plot_df[c] = plot_df[c].astype(str).str.replace("%", "", regex=False).astype(float)

x = plot_df["Time"].values.astype(float)

# ===== 尺寸：8.6 cm × 6.35 cm -> inch =====
cm_to_inch = 1 / 2.54
fig_w = 8.6 * cm_to_inch
fig_h = 6.35 * cm_to_inch

# ===== 字体：尽量锁定 Times New Roman（无则 fallback 到 Times/DejaVu Serif）=====
tnr = font_manager.FontProperties(family="Times New Roman")

# ===== 样式：字号9；字体全黑；坐标轴灰色 =====
axis_gray = "#7a7a7a"
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 9,

    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "legend.frameon": False,

    "axes.linewidth": 0.8,
    "axes.edgecolor": axis_gray,

    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.direction": "out",
    "ytick.direction": "out",

    "svg.fonttype": "path",
})

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

# # ===== 颜色渐变：#c7d4d9 -> #E5A694 =====
# start = np.array(to_rgb("#c7d4d9"))
# end   = np.array(to_rgb("#E29A86"))
# colors = [tuple(start + (end - start) * i / (len(ssp_cols) - 1)) for i in range(len(ssp_cols))]

# ===== 5个SSP分别指定颜色（替换掉原来的渐变代码）=====
ssp_cols = ["ssp1", "ssp2", "ssp3", "ssp4", "ssp5"]

ssp_color = {
    "ssp1": "#c7d6d9",  # green
    "ssp2": "#8694b8",  # olive/green
    "ssp3": "#9570b3",  # purple
    "ssp4": "#FB7D5D",  # yellow/orange
    "ssp5": "#b44f02",  # orange/red
}

# 如果你后面仍然用 colors[i] 的写法，也可以保留这个列表：
colors = [ssp_color[c] for c in ssp_cols]


# 为样条拟合准备：x 必须严格递增
order = np.argsort(x)
x_sorted = x[order]
x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 300)

# ===== 平滑参数 =====
s_val = 50
print(f"Smoothing factor s = {s_val}")

# ================== 不确定性设置（你可按需要改这三个参数） ==================
# 误差随时间累积：2020(起始年)=0，2100(末年)=err_max_2100
y_all = plot_df[ssp_cols].to_numpy(dtype=float)
data_range = np.nanmax(y_all) - np.nanmin(y_all)

err_max_2100 = max(1.0, 0.15 * data_range)  # 2100年的最大误差（单位：百分点），默认取数据范围的10%，且至少1
err_power = 1.0                              # 增长形状：1=线性；>1 后期增长更快；<1 前期增长更快
band_alpha = 0.3                            # 误差带透明度
# ======================================================================

# 把 x_dense 归一化到 [0,1]，保证起点误差=0，终点误差=err_max_2100
t_norm = (x_dense - x_sorted.min()) / (x_sorted.max() - x_sorted.min())
t_norm = np.clip(t_norm, 0, 1)
err_dense = err_max_2100 * (t_norm ** err_power)  # 随年份增加而增大

for i, ssp in enumerate(ssp_cols):
    color = ssp_color[ssp]  # <- 新增这一行
    y = plot_df[ssp].values.astype(float)[order]
    spline = UnivariateSpline(x_sorted, y, k=3, s=s_val)
    y_dense = spline(x_dense)

    # ===== 不确定性误差带（随时间累积）=====
    y_lower = np.maximum(0, y_dense - err_dense)  # 防止下界小于0（你若允许负值可去掉这行的maximum）
    y_upper = y_dense + err_dense
    ax.fill_between(
        x_dense, y_lower, y_upper,
        # color=colors[i],
        color=color,
        alpha=band_alpha,
        linewidth=0,
        zorder=1
    )

    # ===== 趋势线（画在误差带上方）=====
    ax.plot(
        x_dense, y_dense,
        # color=colors[i],
        color=color,
        linewidth=1.2,
        linestyle="-",
        label=ssp.upper(),
        zorder=2
    )

# 轴标签
ax.set_xlabel("Year", fontproperties=tnr, color="black")
ax.set_ylabel("Risk probability change rate (%)", fontproperties=tnr, color="black")

# x刻度：显示原始年份
ax.set_xticks(x_sorted)

# y轴：0-50%，每10%一个刻度
ax.set_ylim(0, 60)
ax.yaxis.set_major_locator(MultipleLocator(10))

# 网格
ax.grid(True, axis="y", linewidth=0.5, alpha=0.25)
ax.grid(False, axis="x")

# 四周边框都显示，并设为灰色
for side in ["left", "bottom", "top", "right"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color(axis_gray)

# tick：文字黑色；刻度线灰色
ax.tick_params(axis="both", labelcolor="black", color=axis_gray)

# 强制 tick label 使用 Times New Roman（如果系统可用）
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontproperties(tnr)
    lab.set_color("black")

# 图例
leg = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    bbox_transform=ax.transAxes,
    ncol=3,
    handlelength=2.0,
    columnspacing=1.0,
    borderaxespad=0.2,
    prop=tnr
)
for t in leg.get_texts():
    t.set_color("black")

fig.tight_layout()

# 输出
fig_path_svg  = os.path.join(out_dir, "risk_pre_ssp_ssp_time_a_ppt_safe_with_uncertainty.svg")
plt.savefig(fig_path_svg, bbox_inches="tight")

plt.show()
plt.close(fig)

print("Figure saved:")
print(fig_path_svg)


In [ ]:
# ================= Figure 3 (right panel only): Province mean baseline vs future =================
# It reads: province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_{ssp}_{year}.csv
# and outputs (6cm x 6cm): risk_pre_ssp_ssp_time_c_province_{ssp}_{year}.svg
import os
import sys
import pickle
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd

CODE_V2_DIR = Path(base_path) / "code"
if str(CODE_V2_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_V2_DIR))

from mgtwr import gwr_sigmoid_utils

fig3_module_path = CODE_V2_DIR / "5_gwr_model_prediction" / "national_fig_ssp_time" / "fig3_province_scatter_plot_updated_with_violin_v5.py"
spec = importlib.util.spec_from_file_location("fig3_province_scatter_plot_updated_with_violin_v5_runtime", fig3_module_path)
fig3_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fig3_module)

CLASSIFICATION_ARTIFACTS_PATH = CODE_V2_DIR / "3_gwr_model_train" / "national" / "GWR" / "gwr_classification_artifacts.pkl"
with CLASSIFICATION_ARTIFACTS_PATH.open("rb") as f:
    gwr_artifacts = pickle.load(f)
transform_metadata = gwr_artifacts["transform_metadata"]

print(f"Loaded fig3 module from: {fig3_module_path}")
print(f"Loaded sigmoid transform metadata from: {CLASSIFICATION_ARTIFACTS_PATH}")
print(transform_metadata)


def to_sigmoid_probability(values):
    probabilities, _ = gwr_sigmoid_utils.gwr_scores_to_probabilities(
        np.asarray(values, dtype=float).reshape(-1),
        transform_metadata=transform_metadata,
    )
    return np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)


def convert_raw_score_df_to_probability_df(df):
    df = df.copy()
    required = ["mean_score_2020", "mean_score_current"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing raw-score columns in CSV: {missing}")

    df["mean_prob_2020"] = to_sigmoid_probability(df["mean_score_2020"])
    df["mean_prob_current"] = to_sigmoid_probability(df["mean_score_current"])

    if "median_score_2020" in df.columns:
        df["median_prob_2020"] = to_sigmoid_probability(df["median_score_2020"])
    if "median_score_current" in df.columns:
        df["median_prob_current"] = to_sigmoid_probability(df["median_score_current"])

    df["delta_mean"] = df["mean_prob_current"] - df["mean_prob_2020"]
    if {"median_prob_2020", "median_prob_current"}.issubset(df.columns):
        df["delta_median"] = df["median_prob_current"] - df["median_prob_2020"]

    base = pd.to_numeric(df["mean_prob_2020"], errors="coerce").to_numpy(dtype=float)
    delta = pd.to_numeric(df["delta_mean"], errors="coerce").to_numpy(dtype=float)
    change_rate = np.full(delta.shape, np.nan, dtype=float)
    valid = np.isfinite(base) & np.isfinite(delta) & (np.abs(base) > 1.0e-12)
    change_rate[valid] = delta[valid] / base[valid]
    df["change_rate"] = change_rate
    df["change_rate_pct"] = change_rate * 100.0
    return df


def average_probability_dfs(dfs):
    if not dfs:
        raise ValueError("dfs is empty")

    join_key = "ADCODE99"
    prepared = []
    for df in dfs:
        tmp = df.copy()
        tmp[join_key] = pd.to_numeric(tmp[join_key], errors="coerce").astype("Int64")
        tmp = tmp.dropna(subset=[join_key]).copy()
        agg_map = {
            "mean_prob_2020": "mean",
            "mean_prob_current": "mean",
        }
        if "NAME_EN_JX" in tmp.columns:
            agg_map["NAME_EN_JX"] = "first"
        if "n_points" in tmp.columns:
            agg_map["n_points"] = "mean"
        if "median_prob_2020" in tmp.columns:
            agg_map["median_prob_2020"] = "mean"
        if "median_prob_current" in tmp.columns:
            agg_map["median_prob_current"] = "mean"
        prepared.append(tmp.groupby(join_key, sort=False).agg(agg_map))

    common_ids = set(prepared[0].index.tolist())
    for g in prepared[1:]:
        common_ids &= set(g.index.tolist())
    common_ids = sorted(common_ids)
    if not common_ids:
        raise ValueError("No common province IDs found across transformed dfs.")

    out = pd.DataFrame(index=pd.Index(common_ids, name=join_key))
    if "NAME_EN_JX" in prepared[0].columns:
        out["NAME_EN_JX"] = prepared[0]["NAME_EN_JX"].reindex(common_ids)
    if "n_points" in prepared[0].columns:
        out["n_points"] = prepared[0]["n_points"].reindex(common_ids)

    out["mean_prob_2020"] = pd.concat(
        [g["mean_prob_2020"].reindex(common_ids) for g in prepared], axis=1
    ).mean(axis=1, skipna=True)
    out["mean_prob_current"] = pd.concat(
        [g["mean_prob_current"].reindex(common_ids) for g in prepared], axis=1
    ).mean(axis=1, skipna=True)

    if all("median_prob_2020" in g.columns for g in prepared):
        out["median_prob_2020"] = pd.concat(
            [g["median_prob_2020"].reindex(common_ids) for g in prepared], axis=1
        ).mean(axis=1, skipna=True)
    if all("median_prob_current" in g.columns for g in prepared):
        out["median_prob_current"] = pd.concat(
            [g["median_prob_current"].reindex(common_ids) for g in prepared], axis=1
        ).mean(axis=1, skipna=True)

    out["delta_mean"] = out["mean_prob_current"] - out["mean_prob_2020"]
    if {"median_prob_2020", "median_prob_current"}.issubset(out.columns):
        out["delta_median"] = out["median_prob_current"] - out["median_prob_2020"]

    base = pd.to_numeric(out["mean_prob_2020"], errors="coerce").to_numpy(dtype=float)
    delta = pd.to_numeric(out["delta_mean"], errors="coerce").to_numpy(dtype=float)
    change_rate = np.full(delta.shape, np.nan, dtype=float)
    valid = np.isfinite(base) & np.isfinite(delta) & (np.abs(base) > 1.0e-12)
    change_rate[valid] = delta[valid] / base[valid]
    out["change_rate"] = change_rate
    out["change_rate_pct"] = change_rate * 100.0
    return out.reset_index()


# 5个情景 + 年份
cases_cfg = [
    ("ssp1", "2040"),
    ("ssp2", "2080"),
    ("ssp3", "2100"),
    ("ssp4", "2100"),
    ("ssp5", "2100"),
]

save_violin = True
plot_cases = []
for ssp, ssp_time in cases_cfg:
    fname = f"province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_{ssp}_{ssp_time}.csv"
    in_path = os.path.join(out_dir, ssp, ssp_time, fname)
    if not os.path.exists(in_path):
        raise FileNotFoundError(f"Missing input CSV: {in_path}")

    raw_df = fig3_module.load_province_change_csv(in_path)
    prob_df = convert_raw_score_df_to_probability_df(raw_df)
    plot_cases.append((ssp, ssp_time, in_path, prob_df))

first_case_path = plot_cases[0][2]
first_prob_df = plot_cases[0][3]
print(f"First input CSV: {first_case_path}")
print("First input CSV columns:")
print(pd.read_csv(first_case_path, nrows=1).columns.tolist())
print("First transformed probability range:")
print(
    float(np.nanmin(first_prob_df["mean_prob_2020"])),
    float(np.nanmax(first_prob_df["mean_prob_2020"])),
    float(np.nanmin(first_prob_df["mean_prob_current"])),
    float(np.nanmax(first_prob_df["mean_prob_current"])),
)

fig_paths = []
for ssp, ssp_time, in_path, prob_df in plot_cases:
    save_path = os.path.join(out_dir, ssp, ssp_time, f"risk_pre_ssp_ssp_time_c_province_{ssp}_{ssp_time}.svg")
    saved = fig3_module.make_fig3_scatter_only(
        prob_df,
        ssp=ssp,
        year=str(ssp_time),
        save_path=save_path,
        width_cm=6,
        height_cm=6,
    )
    if save_violin:
        violin_path = os.path.splitext(save_path)[0] + "_violin.svg"
        fig3_module.make_fig3_violin_only(
            prob_df,
            ssp=ssp,
            year=str(ssp_time),
            save_path=violin_path,
            width_cm=6,
            height_cm=6,
            value_col="delta_mean",
        )
    fig_paths.append(saved)

avg_df = average_probability_dfs([item[3] for item in plot_cases])
avg_dir = os.path.join(out_dir, "mean", "mean_5cases")
os.makedirs(avg_dir, exist_ok=True)
avg_path = os.path.join(avg_dir, "risk_pre_ssp_ssp_time_c_province_mean_mean_5cases.svg")
saved_avg = fig3_module.make_fig3_scatter_only(
    avg_df,
    ssp="mean",
    year="mean_5cases",
    save_path=avg_path,
    width_cm=6,
    height_cm=6,
)
if save_violin:
    avg_violin_path = os.path.splitext(avg_path)[0] + "_violin.svg"
    fig3_module.make_fig3_violin_only(
        avg_df,
        ssp="mean",
        year="mean_5cases",
        save_path=avg_violin_path,
        width_cm=6,
        height_cm=6,
        value_col="delta_mean",
    )
fig_paths.append(saved_avg)

print("All figures saved:")
for p in fig_paths:
    print(p)
